# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fmarryam70-ux/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



My lane is Lane 2: Refresh / Content Opportunity Scoring — a **ranking** question
("which pages should a reviewer look at first?"), not a plain yes/no question. Per
the `training-honest-models` skill, ranking needs *scores*, not just labels, so the
right move is: any classifier's probability, evaluated at precision@K.

I'm starting with **Logistic Regression** (readable, gives a clean coefficient story)
and comparing it against a **Decision Tree** (depth 4, so it can still be printed and
read) and a **Random Forest** (stronger, but only worth it if it actually beats the
simpler models). This follows the toolkit directly: "readable → stronger," and
"simplicity is a feature — add complexity only when the comparison earns it."

All three are evaluated the same way my Week-4 rule baseline was judged: rank every
test-set page by score, take the top 20 / top 50, and measure precision@K — the same
metric a reviewer actually cares about (how many of the pages I send them are truly
declining).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/fmarryam70-ux/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same label as Week 4
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df)}, base rate (declining): {df['is_declining_label'].mean():.3f}")

Rows: 30000, base rate (declining): 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



**Client-grouped split** (80/20 by `client_id`, not by row). The data dictionary is
explicit that `client_id` is a pseudonym for grouping/joins only — never a feature —
and that it should be used for **client-holdout splits**.

This is the same client-holdout idea used to build Week 4's baseline queue, so the
comparison in Section 3 is apples-to-apples.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(df, groups=df["client_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

overlap = set(train["client_id"]) & set(test["client_id"])

print(f"train rows: {len(train)}")
print(f"test rows: {len(test)}")
print(f"client overlap: {len(overlap)}")

train rows: 23837
test rows: 6163
client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



I recompute the Week-4 rule score on the **test split only** and compare it with:

- Logistic Regression
- Decision Tree
- Random Forest

The comparison uses:

- Precision@20
- Precision@50

The same split and the same evaluation metric are used for every model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ---- Week-4 baseline score, recomputed on the test split only ----
def baseline_score(d):
    visible = (d["impressions_90d"] >= 500).astype(int)
    stale = (d["days_since_last_update"] >= 180).astype(int)
    low_ctr_visible = ((d["avg_position"] > 0) & (d["avg_position"] <= 20)
                        & (d["ctr"] < 0.5) & visible).astype(int)
    return (0.55 * stale * visible * np.log1p(d["impressions_90d"])
            + 0.45 * low_ctr_visible * np.log1p(d["impressions_90d"]))

test["baseline_score"] = baseline_score(test)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true, "score": scores})
    top = frame.sort_values("score", ascending=False).head(k)
    return float(top["y"].mean())

# ---- Feature engineering (no leakage columns, explicit missingness flags) ----
MISSING_PRONE = ["word_count", "char_count", "search_volume", "competition", "cpc", "scroll_rate"]
for d in (train, test):
    for col in MISSING_PRONE:
        d[f"has_{col}"] = d[col].notna().astype(int)
        d[col] = d[col].fillna(0)
    d["log_impressions_90d"] = np.log1p(d["impressions_90d"])
    d["log_clicks_90d"] = np.log1p(d["clicks_90d"])

NUMERIC = ["log_impressions_90d", "log_clicks_90d", "days_with_impressions", "days_with_sessions",
           "content_age_days", "days_since_last_update", "ctr", "avg_position", "engagement_rate",
           "scroll_rate", "word_count", "char_count", "search_volume", "competition", "cpc"] + \
          [f"has_{c}" for c in MISSING_PRONE]
CATEGORICAL = ["content_type", "main_intent", "age_tier", "freshness_tier", "impression_tier", "position_tier"]

X_train, y_train = train[NUMERIC + CATEGORICAL], train["is_declining_label"]
X_test, y_test = test[NUMERIC + CATEGORICAL], test["is_declining_label"]

pre = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])

models = {
    "logistic_regression": Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "decision_tree": Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=SEED))]),
    "random_forest": Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=SEED, n_jobs=-1))]),
}

model_scores = {}
rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    model_scores[name] = scores
    rows.append({"model": name,
                 "precision_at_20": precision_at_k(y_test, scores, 20),
                 "precision_at_50": precision_at_k(y_test, scores, 50)})

rows.append({"model": "baseline_rules",
             "precision_at_20": precision_at_k(y_test, test["baseline_score"], 20),
             "precision_at_50": precision_at_k(y_test, test["baseline_score"], 50)})
rows.append({"model": "base_rate (random)",
             "precision_at_20": y_test.mean(), "precision_at_50": y_test.mean()})

comparison = pd.DataFrame(rows).sort_values("precision_at_50", ascending=False).reset_index(drop=True)
print(comparison.round(3).to_string(index=False))


              model  precision_at_20  precision_at_50
logistic_regression            0.850            0.740
      decision_tree            0.450            0.580
      random_forest            0.550            0.580
 base_rate (random)            0.511            0.511
     baseline_rules            0.400            0.420


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



Logistic Regression comes out on top.

It beats:

- Decision Tree
- Random Forest
- Week-4 Rule Baseline

Permutation Importance is used to interpret the model.

In [ ]:
from sklearn.inspection import permutation_importance

best_name = comparison.iloc[0]["model"]
print("Best model by precision@50:", best_name)

if best_name in models:
    best_pipe = models[best_name]
    perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=10,
                                   random_state=SEED, scoring="roc_auc", n_jobs=-1)
    importances = pd.Series(perm.importances_mean, index=NUMERIC + CATEGORICAL) \
                    .sort_values(ascending=False)
    print("\nTop 8 features by permutation importance (ROC AUC drop when shuffled):")
    print(importances.head(8).round(4))

    test_c = test.copy()
    test_c["model_score"] = model_scores[best_name]
    top50 = test_c.sort_values("model_score", ascending=False).head(50)
    wrong = top50[top50["is_declining_label"] == 0]

    print(f"\nWrong picks in the model's top 50: {len(wrong)} of 50")
    print("(Week-4 baseline wrong picks in its own top 50 for comparison: "
          f"{50 - int(precision_at_k(y_test, test['baseline_score'], 50) * 50)})")

    print("\n3 concrete wrong cases:")
    for _, row in wrong.head(3).iterrows():
        print(f"- {row['content_id']}: model_score={row['model_score']:.2f}, "
              f"trend={row['trend_direction']}, impressions_90d={row['impressions_90d']}, "
              f"avg_position={row['avg_position']}, ctr={row['ctr']}")
        print("  why it's hard: high impressions/clicks and a weak-looking position or CTR "
              "still resemble a declining page's profile, but the page's trend is actually "
              "'up' or 'stable' — the model can't see *why* (seasonality, a recent refresh, "
              "a promo) from these features alone, so it over-indexes on the visibility signal.")


Best model by precision@50: logistic_regression

Top 8 features by permutation importance (ROC AUC drop when shuffled):
log_impressions_90d    0.0765
log_clicks_90d         0.0573
impression_tier        0.0363
days_with_sessions     0.0266
avg_position           0.0244
content_age_days       0.0174
scroll_rate            0.0072
char_count             0.0052
dtype: float64

Wrong picks in the model's top 50: 13 of 50
(Week-4 baseline wrong picks in its own top 50 for comparison: 29)

3 concrete wrong cases:
- content_7be5f150dc65: model_score=0.96, trend=up, impressions_90d=290, avg_position=5.9, ctr=0.0
  why it's hard: high impressions/clicks and a weak-looking position or CTR still resemble a declining page's profile, but the page's trend is actually 'up' or 'stable' — the model can't see *why* (seasonality, a recent refresh, a promo) from these features alone, so it over-indexes on the visibility signal.
- content_374e795aab68: model_score=0.93, trend=stable, impressions_90d=235, av

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.